# Gemini によるコンテキスト キャッシュ（Context Caching）

## 学習目標

1.  コンテキスト キャッシュ（Context Caching）の概念と、大規模言語モデルを扱う際におけるそのメリットを理解する。
2.  Agent Platform SDK を使用して、Gemini モデルでキャッシュされたコンテンツを作成および利用する方法を学ぶ。
3.  キャッシュされたコンテンツを使用した場合と、ゼロからコンテンツを生成した場合のパフォーマンスを比較し、速度とコストの利点を確認する。

## 概要
このノートブックでは、大規模で一貫性のある初期コンテキストを含むリクエストを最適化するために設計された、Gemini API のコンテキスト キャッシュ（Context Caching）機能について確認します。

## 基本セットアップ

In [ ]:
import datetime

from google import genai
from google.genai import types

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
MODEL = "gemini-3.5-flash"

## コンテキスト キャッシュ

このノートブックの 2 番目のセクションでは、Agent Platform の Gemini モデルでコンテキスト キャッシュを使用する方法を説明します。

コンテキスト キャッシュを使用すると、研究論文、長時間の動画や音声などの処理済みコンテンツをシステム指示とともに保存できるため、毎回再処理する必要がなくなります。<br>
モデルにクエリを実行する際、保存されたコンテキストを活用できるため、応答時間の短縮とリソース消費の削減につながります。これは、大規模なドキュメントを扱う場合や、複数のクエリにわたって同じコンテキストを使用する場合に特に役立ちます。

### コンテンツの定義

ここでは、Google Cloud Storage に保存されている PDF 形式の研究論文への参照をそれぞれ含む `Part` オブジェクトのリストとして `contents` 変数を定義します。<br>
これらは、コンテキスト キャッシュに使用される論文です。

In [ ]:
system_instruction = """
You are an expert researcher. You always stick to the facts in the sources provided, and never make up new facts.
Now look at these research papers, and answer the following questions.
"""

contents = [
    types.Part.from_uri(
        file_uri="gs://asl-public-data/data/generative-ai/pdf/2312.11805v3.pdf",
        mime_type="application/pdf",
    ),
    types.Part.from_uri(
        file_uri="gs://asl-public-data/data/generative-ai/pdf/2403.05530.pdf",
        mime_type="application/pdf",
    ),
]

### コンテキスト キャッシュの作成

キャッシュされたコンテンツを作成しましょう。`client.caches.create` を使用して、指定されたパラメータでキャッシュを設定します。パラメータは次のとおりです。

*   `model`: 使用する Gemini モデルを指定します（この場合は "gemini-3.5-flash"）。
*   `config`: 基本構成。以下が含まれます。
    *   `system_instruction`: モデルの動作に関する指示を設定します。
    *   `contents`: キャッシュに保存する実際のドキュメントやその他のデータ。
    *   `ttl`: キャッシュの有効期間（Time-to-Live、この場合は 60 分）。これを過ぎるとキャッシュは期限切れになります。
    *   `display_name`: 簡単に識別するための名前。

このセルの出力は、後でキャッシュされたコンテンツを取得するために使用される一意の識別子 `cached_content.name` です。

In [ ]:
client = genai.Client(vertexai=True, location="global")

cached_content = client.caches.create(
    model=MODEL,
    config=types.CreateCachedContentConfig(
        system_instruction=system_instruction,
        contents=contents,
        ttl="3600s",
        display_name="example-cache",
    ),
)

print(cached_content.name)

作成されたコンテキスト キャッシュを確認してみましょう。

In [ ]:
for cache in client.caches.list():
    print(cache)

### キャッシュされたコンテキストなしでの生成

比較のため、まずはキャッシュされたコンテンツを **使用せずに** 回答を生成し、処理時間を記録します。

In [ ]:
%%time
response = client.models.generate_content(
    model=MODEL, contents=contents + ["What are the papers about?"]
)

print(response.text)

### キャッシュされたコンテキストを使用した生成

それでは、キャッシュされたコンテンツを使用して回答を生成してみましょう。`cached_content` パラメータは、作成されたキャッシュコンテンツを参照します。

In [ ]:
%%time
response = client.models.generate_content(
    model=MODEL,
    contents="What are the papers about?",
    config=types.GenerateContentConfig(cached_content=cached_content.name),
)
print(response.text)

出力結果から、処理時間が大幅に短縮されたことが明確にわかります。

このパフォーマンスの向上は、コンテキスト情報の量が増加するにつれてより顕著になります。処理済みのコンテキストを保存して再利用することで、特に大規模なコンテキストにおいて大幅な効率向上が達成されます。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.